# 🏦 Comparativa Estratégica: Flota Propia vs. Subcontratación (Make or Buy)

Este análisis cuantifica el impacto financiero de operar bajo un modelo de flota propia (basado en el TCO interno) frente a la contratación de servicios de transporte externos a tarifas de mercado.

### 📐 Modelado del Margen de Ahorro Operativo

El ahorro generado por cada kilómetro recorrido con flota propia en comparación con un proveedor externo se define como:

$$ \Delta \tau = \tau_{ext} - \tau_{int} $$

Donde:
- $\tau_{ext}$: Tarifa de mercado del proveedor externo (€/km).
- $\tau_{int}$: Tarifa técnica interna calculada mediante TCO (€/km).

El **Ahorro Operativo Mensual ($S_m$)** se calcula como:

$$ S_m = \sum_{i=1}^{n} d_i \times \Delta \tau $$

Donde $d_i$ es la distancia total recorrida en el periodo.

In [1]:
import sys
import os
import pandas as pd
import plotly.graph_objects as go

# Añadir el directorio raíz al path para importar logistic_core
sys.path.append(os.path.abspath(os.path.join('..', '..', '..')))

from logistic_core.utils.external_cost_analyst import ExternalCostAnalyst
import logistic_core.config as config

# Inicialización del analista con datos reales de configuración
analyst = ExternalCostAnalyst(
    internal_rate=config.INTERNAL_OPERATIONAL_TCO_RATE,
    external_rate=config.EXTERNAL_PROVIDER_RATE_PER_KM
)

2026-04-13 20:02:05,398 | WARNING  | config | GOOGLE_MAPS_API_KEY no está configurada. Se usará estimación Haversine.


2026-04-13 20:02:05,399 | INFO     | root | ExternalCostAnalyst inicializado. Interno: 1.5 €/km, Externo: 1.7 €/km


## 1. Simulación de un Tramo de Larga Distancia

Evaluamos un viaje de ida y vuelta (Roundtrip) de 800 km.

In [2]:
distancia_sim = 800
resultado = analyst.analyze_leg(linehaul_distance=distancia_sim)

print(f"--- ANÁLISIS DE TRAMO ({distancia_sim} km) ---")
print(f"Coste Externo: {resultado['external_cost']:,.2f} €")
print(f"Coste Interno: {resultado['internal_cost']:,.2f} €")
print(f"Ahorro Directo: {resultado['savings']:,.2f} €")
print(f"Margen de Ahorro: {(resultado['savings']/resultado['external_cost'])*100:.1f}%")

--- ANÁLISIS DE TRAMO (800 km) ---
Coste Externo: 1,360.00 €
Coste Interno: 1,200.00 €
Ahorro Directo: 160.00 €
Margen de Ahorro: 11.8%


## 2. Proyección de Ahorro Anual por Flota

Visualizamos el ahorro potencial dependiendo del kilometraje anual total de la flota.

In [3]:
kms = [100_000, 500_000, 1_000_000, 5_000_000, 10_000_000]
outsourcing = [k * analyst.external_rate for k in kms]
inhouse = [k * analyst.internal_rate for k in kms]

fig = go.Figure()
fig.add_trace(go.Scatter(x=kms, y=outsourcing, name="Coste Outsourcing", line=dict(color='#2c3e50')))
fig.add_trace(go.Scatter(x=kms, y=inhouse, name="Coste Flota Propia", line=dict(color='#27ae60')))
fig.update_layout(title="Proyección de Costeo: Interno vs Externo", xaxis_title="Km Anuales Totales", yaxis_title="Coste Total (€)")
fig.show()

### Conclusión Forense
Como se demuestra, un ahorro de **0.15 €/km** (basado en la diferencia entre el TCO de 1.50 €/km y la tarifa externa de 1.65 €/km) escala rápidamente a ahorros de seis cifras para flotas con altas intensidades de uso. Estos ahorros justifican la inversión en CAPEX para camiones propios bajo un horizonte de 5 años.